<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/19-interpretability-robustness-fairness-privacy.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Interpretability, Robustness, Fairness, and Privacy**

A model can be accurate on a benchmark and still be unsuitable for use. It may depend on a leakage feature, fail when the environment changes, concentrate errors on a small population, memorize training records, or produce decisions that no responsible person can contest. These are not rare edge cases added after “the real modeling work.” They determine whether the measured performance can be trusted in the setting where the model acts.

This chapter treats four requirements as distinct:

- **Interpretability** asks what behavior a model has learned and whether an explanation is faithful enough for its intended audience and decision.
- **Robustness** asks how performance changes under perturbations, distribution shift, missingness, adversarial behavior, and operational failure.
- **Fairness** asks how benefits, burdens, errors, and decision rules are distributed across people, especially groups exposed to a relevant harm.
- **Privacy** asks what information about a person, record, client, or population can be inferred from data, parameters, gradients, and outputs.

<div class="diagram-scroll">

![Interpretability, robustness, fairness, and privacy are separate reliability dimensions.](assets/trustworthy-ml-dimensions.svg){fig-alt="Four boxes summarize interpretability, robustness, fairness, and privacy, each with a different evaluation question."}

</div>

Passing one dimension does not certify another. A transparent linear model can be discriminatory; a differentially private model can be inaccurate for a subgroup; a robust classifier can still reveal membership; and a fair aggregate metric can hide arbitrary local behavior. A useful review therefore begins with a **claim-evidence map**:

| Claim | Evidence that is relevant | Evidence that is insufficient by itself |
|---|---|---|
| “The model is understandable” | A specified audience, explanation target, fidelity check, stability check, and known limitations | A colorful feature-importance plot |
| “The model is robust” | Named shift or attacker, severity range, slice-level degradation, uncertainty, and fallback behavior | Clean test accuracy |
| “The model is fair” | A harm model, affected groups, justified metric, uncertainty, and impact of mitigation | One parity number chosen after inspection |
| “The model is private” | Privacy unit, neighboring relation, threat model, release interface, accounting, and implementation audit | Data staying on a local device |

The model is only one component of a **sociotechnical system**. Data collection, labels, user behavior, thresholds, human review, access control, appeal, and feedback loops can dominate system outcomes. Consequently, evaluation must follow the complete decision process rather than stop at a fitted estimator.

### **Interpretability and Explainability**

The terms *interpretability* and *explainability* are used inconsistently, so the intended claim should be stated directly. In this chapter:

- **Interpretability** is the degree to which a person can understand a model's relevant behavior, such as how variables affect predictions or which rule produced a decision.
- **Explanation** is an artifact produced for a specific prediction, model behavior, or audience.
- **Transparency** concerns access to and clarity about data, model design, training, ownership, and decision policy.
- **Faithfulness** measures whether an explanation accurately reflects the behavior of the model being explained.
- **Plausibility** measures whether an explanation sounds reasonable to a person.

Faithfulness and plausibility are not equivalent. A plausible explanation can rationalize a prediction using concepts the model did not actually use. Conversely, a faithful explanation can expose an undesirable dependency that is hard to communicate. Explanation quality must therefore be tested against a precise target:

1. **Object:** prediction, score, ranking, representation, training example, or policy.
2. **Audience:** developer, domain expert, auditor, operator, or affected person.
3. **Purpose:** debugging, scientific understanding, validation, recourse, compliance, or monitoring.
4. **Fidelity region:** one point, a local neighborhood, a subgroup, or the whole input distribution.
5. **Output scale:** probability, log-odds, raw score, class, or utility.

An explanation can be correct for one of these contracts and misleading for another. For example, a local linear approximation may explain the raw score near one applicant but say nothing about global monotonicity or the causal effect of changing an applicant attribute.

#### **Intrinsic and Post-Hoc Interpretation**

An **intrinsically interpretable model** exposes its decision structure directly. Examples include a sparse linear model, a short decision list, a shallow tree, a generalized additive model, or a monotonic scorecard. Interpretability is not a property of the algorithm name alone. A linear model with 50,000 correlated features and opaque preprocessing is not meaningfully inspectable, while a carefully constrained nonlinear model can be.

A **post-hoc explanation** analyzes a fitted model after training. It may perturb inputs, fit a local surrogate, decompose a prediction, retrieve influential training points, visualize activations, or search for a counterfactual. Post-hoc methods are valuable when the predictive model is complex, but they create a second estimation problem: the explanation itself has assumptions, approximation error, randomness, and a reference distribution.

The central validation question is:

> If the explanation says a feature or rule matters, does changing or removing that component alter the model in the predicted way on the stated region?

Useful checks include perturbation tests, surrogate fidelity, repeated runs with different seeds, bootstrap uncertainty, sensitivity to the background dataset, and comparison with a known synthetic ground truth. Agreement between two explanation tools is encouraging but not proof, because both may share the same invalid independence assumption.

#### **Global and Local Explanations**

A **global explanation** summarizes behavior over a distribution: overall feature importance, a response curve, a compact surrogate tree, a rule set, or subgroup behavior. A **local explanation** describes one prediction or a small neighborhood: feature contributions, a local surrogate, a counterfactual, or influential examples.

<div class="diagram-scroll">

![Global versus local and intrinsic versus post-hoc are separate explanation dimensions.](assets/explanation-scope-map.svg){fig-alt="A two-by-two conceptual map distinguishes global from local scope and intrinsic from post-hoc explanation mechanisms."}

</div>

The data distribution is part of a global explanation. “Feature $j$ is important” means important under a particular model, metric, perturbation, and population. If the deployment mix changes, so can the explanation. A local explanation also needs a neighborhood definition: distance in raw input space may be meaningless for text, images, encoded categories, or correlated clinical measurements.

Interpretation should not be promoted to causation. Most feature-attribution methods explain $f(x)$, not $Y(do(X_j=x_j'))$. If income and education are correlated, replacing one while holding the other fixed can create an implausible person; even a realistic change in a model input does not establish the outcome that a real intervention would cause.

**Comparison.** Intrinsic models offer direct structural evidence but may require capacity constraints. Post-hoc methods support complex models but need fidelity tests. Global explanations support validation and governance; local explanations support debugging and individual review. A responsible analysis often needs both, because a globally acceptable model can contain locally harmful decisions and a locally plausible explanation can conceal a poor global policy.


### **Model-Specific Interpretation**

Model-specific methods use the estimator's mathematical structure. They are usually faster and more exact than black-box perturbation methods, but their meaning still depends on feature construction, output scale, regularization, and dependence among variables.

#### **Linear Coefficients and Tree Structure**

For linear regression,

$$
\hat y=\beta_0+\sum_{j=1}^{p}\beta_jx_j,
$$

$\beta_j$ is the change in the fitted output for a one-unit increase in $x_j$ while the other encoded features are held fixed. That phrase contains several caveats:

- coefficients are not comparable across features with different units unless a meaningful standardization is used;
- one-hot coefficients are relative to an omitted reference category;
- interactions make a feature's effect depend on other variables;
- strong collinearity can make individual coefficients unstable even when predictions are stable;
- regularization changes coefficients by optimizing prediction plus a penalty;
- “holding other variables fixed” is associational and can describe impossible combinations.

For logistic regression,

$$
\log\frac{P(Y=1\mid x)}{1-P(Y=1\mid x)}
=\beta_0+\sum_j\beta_jx_j.
$$

Therefore $\exp(\beta_j)$ is a **conditional odds ratio** for a one-unit change, not a probability difference. A fixed log-odds change has a small probability effect near 0 or 1 and a larger effect near 0.5:

$$
\frac{\partial P(Y=1\mid x)}{\partial x_j}
=\beta_j\,p(x)\bigl(1-p(x)\bigr).
$$

A decision tree is interpreted by following the exact path from the root to a leaf. Each split narrows the region of feature space, and the leaf prediction summarizes training observations in that region. Global structure can be inspected through depth, number of leaves, split thresholds, support per leaf, and repeated use of features. However, impurity-based importance is biased toward features with many candidate split points and can distribute credit unpredictably among substitutes.

<details>
<summary><strong>Python: audit linear coefficients and an exact tree decision path</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

rng = np.random.default_rng(19)
X, y = make_classification(
    n_samples=1600,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    class_sep=1.1,
    random_state=19,
)

# Put feature 0 on a much larger numerical scale. Raw coefficient magnitude
# will shrink even though the information carried by the feature is unchanged.
X_scaled_units = X.copy()
X_scaled_units[:, 0] *= 100.0

raw_model = LogisticRegression(max_iter=2000).fit(X_scaled_units, y)
standardizer = StandardScaler().fit(X_scaled_units)
standard_model = LogisticRegression(max_iter=2000).fit(
    standardizer.transform(X_scaled_units), y
)

print("Raw coefficients:", np.round(raw_model.coef_[0], 4))
print("Standardized coefficients:", np.round(standard_model.coef_[0], 4))
print("Standardized odds ratios:", np.round(np.exp(standard_model.coef_[0]), 3))

# A shallow tree exposes the exact rule used for a selected prediction.
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=60, random_state=19)
tree.fit(X, y)
sample = X[[0]]
leaf_id = tree.apply(sample)[0]
path_nodes = tree.decision_path(sample).indices

print("\nTree rules:\n", export_text(tree, feature_names=[f"x{i}" for i in range(4)]))
print("Selected sample:", np.round(sample[0], 3))
print("Visited node IDs:", path_nodes.tolist())
print("Leaf ID and predicted probability:", leaf_id, np.round(tree.predict_proba(sample)[0], 3))
```

</details>

The standardized coefficient compares a one-standard-deviation movement under the training distribution; it does not make the feature causal or actionable. The tree path is exact for that fitted tree, but a small data perturbation can change the tree structure. Stability should be evaluated by refitting across folds or bootstrap samples and comparing selected features, thresholds, and predictions.

Model-specific interpretation is strongest when constraints are chosen deliberately:

| Model design | Interpretability benefit | Remaining risk |
|---|---|---|
| Sparse linear or logistic model | Few additive terms | Correlation and nonlinear misspecification |
| Generalized additive model | Smooth one-feature functions | Interactions must be added explicitly |
| Monotonic model | Enforces a directional relationship | Direction can be wrong or incomplete |
| Shallow tree or decision list | Human-readable rules | Instability and coarse boundaries |
| Scorecard with bounded points | Operational transparency | Discretization and threshold effects |

**Summary.** Read coefficients on their correct scale, inspect the entire preprocessing pipeline, and distinguish exact structural interpretation from stability across possible training samples. A simple model is preferable when its constraints preserve adequate utility and make the consequential behavior auditable; simplicity should be measured in the representation that a human actually sees.


### **Model-Agnostic Interpretation**

Model-agnostic methods treat a fitted predictor as a function that can be queried. This makes them broadly reusable, but “agnostic” does not mean assumption-free. Every perturbation method defines a synthetic data distribution, and every local method defines a neighborhood. The result explains model behavior under those design choices.

#### **Permutation Importance**

Permutation importance measures how much predictive performance deteriorates when the association between one feature and the target is broken in an evaluation set. For loss $L$, model $f$, and a random permutation $\pi$ of feature $j$,

$$
I_j
=
\mathbb E\!\left[
L\!\left(Y,f(X_{-j},X_j^\pi)\right)
\right]
-
\mathbb E[L(Y,f(X))].
$$

For a score that is better when larger, the subtraction is reversed. The method answers:

> How much does this fitted model rely on information in feature $j$ under this evaluation distribution and this permutation scheme?

It does **not** measure intrinsic value, causal effect, or how well a new model would perform if the feature had never been collected. Retraining after feature removal answers a different question because the model can learn substitutes.

Permutation importance should be computed on validation or test data rather than training data. Multiple permutations provide a Monte Carlo distribution. A near-zero value can mean the feature is useless, the model ignored it, a correlated feature substituted for it, or the metric is insensitive to the errors it affects.

<div class="diagram-scroll">

![Individual permutation can understate importance when correlated features substitute for one another.](assets/correlated-feature-importance.svg){fig-alt="Two correlated features feed a model; permuting one leaves its substitute available, while grouped permutation destroys their shared information."}

</div>

Marginal permutation also creates combinations that may not occur naturally. **Grouped permutation** can measure a correlated feature set; **conditional permutation** attempts to sample $X_j$ given $X_{-j}$, preserving dependence but changing the estimand and requiring an additional conditional model.

<details>
<summary><strong>Python: expose masked importance under correlated substitutes</strong></summary>

```python
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(3)
n = 4000
latent = rng.normal(size=n)

# x1 and x2 are noisy measurements of the same latent signal.
x1 = latent + rng.normal(scale=0.12, size=n)
x2 = latent + rng.normal(scale=0.12, size=n)
x3 = rng.normal(size=n)
y = 2.5 * latent + 0.4 * x3 + rng.normal(scale=0.5, size=n)
X = np.column_stack([x1, x2, x3])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, random_state=3
)
model = RandomForestRegressor(
    n_estimators=160, min_samples_leaf=8, random_state=3, n_jobs=1
).fit(X_train, y_train)

baseline = mean_squared_error(y_test, model.predict(X_test))

def permuted_increase(columns, repeats=20):
    increases = []
    for _ in range(repeats):
        changed = X_test.copy()
        order = rng.permutation(len(changed))
        # Use one shared row permutation so grouped features retain their
        # within-row relationship while their link with y is destroyed.
        changed[:, columns] = changed[order][:, columns]
        loss = mean_squared_error(y_test, model.predict(changed))
        increases.append(loss - baseline)
    return np.mean(increases), np.std(increases, ddof=1)

for columns, label in [([0], "x1"), ([1], "x2"), ([2], "x3"), ([0, 1], "{x1, x2}")]:
    mean_increase, sd_increase = permuted_increase(columns)
    print(f"{label:8s}: MSE increase = {mean_increase:.3f} +/- {sd_increase:.3f}")
```

</details>

The grouped result is not the sum of individual importances because features interact and substitute for one another. Importance values are therefore not additive budgets unless the explanation method explicitly guarantees additivity on the chosen output scale.

#### **Partial Dependence, ICE, and ALE**

Feature-effect plots summarize how a fitted prediction changes as a feature varies.

For feature subset $S$ and complement $C$, the empirical **partial dependence function** is

$$
\widehat{PD}_S(x_S)
=
\frac{1}{n}\sum_{i=1}^{n}f(x_S,x_C^{(i)}).
$$

It replaces $X_S$ with the grid value for every row, predicts, and averages. This estimates a model-based marginal response under a distribution that combines $x_S$ with observed $x_C$. If $X_S$ and $X_C$ are dependent, some combinations can be unsupported, so the curve may summarize extrapolation.

**Individual Conditional Expectation (ICE)** retains each row:

$$
\widehat{ICE}_i(x_S)=f(x_S,x_C^{(i)}).
$$

The PDP is the average of ICE curves. Parallel ICE curves suggest a mostly additive effect; crossing or different slopes reveal interactions and heterogeneous model behavior that averaging can hide. Centered ICE subtracts each curve's value at a reference point to emphasize shape rather than baseline level.

**Accumulated Local Effects (ALE)** avoid global replacement. For one continuous feature,

$$
ALE_j(x)
=
\int_{z_0}^{x}
\mathbb E\!\left[
\frac{\partial f(X)}{\partial x_j}
\middle| X_j=z
\right]dz
-
\text{centering constant}.
$$

In practice, the range is partitioned into bins. For observations naturally inside a bin, predictions at the bin's upper and lower boundaries are differenced. The mean local difference is accumulated across bins and centered. ALE reduces unrealistic extrapolation under correlated features, although sparse bins, discontinuous variables, and a poor fitted model can still produce unstable curves.

<div class="diagram-scroll">

![PDP, ICE, and ALE use different averaging operations.](assets/dependence-methods.svg){fig-alt="Three panels compare partial dependence, individual conditional expectation, and accumulated local effects, including their assumptions and interpretation."}

</div>

<details>
<summary><strong>Python: compute PDP, ICE, and first-order ALE from model queries</strong></summary>

```python
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

rng = np.random.default_rng(11)
n = 2400
x1 = rng.normal(size=n)
x2 = 0.75 * x1 + rng.normal(scale=0.65, size=n)
y = np.sin(1.4 * x1) + 0.7 * x1 * x2 + rng.normal(scale=0.18, size=n)
X = np.column_stack([x1, x2])

model = HistGradientBoostingRegressor(
    max_depth=4, max_iter=180, learning_rate=0.06, random_state=11
).fit(X, y)

grid = np.quantile(x1, np.linspace(0.05, 0.95, 9))

# PDP replaces x1 for every observation and averages predictions.
pdp = []
ice = np.empty((5, len(grid)))
for grid_index, value in enumerate(grid):
    replaced = X.copy()
    replaced[:, 0] = value
    pdp.append(model.predict(replaced).mean())

    local_rows = X[:5].copy()
    local_rows[:, 0] = value
    ice[:, grid_index] = model.predict(local_rows)

# First-order ALE using quantile bins.
edges = np.unique(np.quantile(x1, np.linspace(0, 1, 11)))
bin_ids = np.clip(np.digitize(x1, edges[1:-1]), 0, len(edges) - 2)
local_effects = np.zeros(len(edges) - 1)
bin_counts = np.zeros(len(edges) - 1, dtype=int)

for bin_id in range(len(edges) - 1):
    rows = np.flatnonzero(bin_ids == bin_id)
    bin_counts[bin_id] = len(rows)
    lower = X[rows].copy()
    upper = X[rows].copy()
    lower[:, 0] = edges[bin_id]
    upper[:, 0] = edges[bin_id + 1]
    local_effects[bin_id] = np.mean(model.predict(upper) - model.predict(lower))

ale_at_bins = np.cumsum(local_effects)
ale_at_bins -= np.average(ale_at_bins, weights=bin_counts)

print("grid:", np.round(grid, 2))
print("PDP:", np.round(pdp, 3))
print("first two ICE curves:\n", np.round(ice[:2], 3))
print("ALE bin effects:", np.round(ale_at_bins, 3))
print("observations per ALE bin:", bin_counts.tolist())
```

</details>

PDP, ICE, and ALE are descriptive probes of $f$. A rising curve means the model score rises under the method's input manipulation, not that intervening on the real-world variable will improve the outcome. Plot rug marks, quantile ranges, or bin counts so readers can see where the model has support.

**Comparison.**

| Method | Scope | Main strength | Main failure mode |
|---|---|---|---|
| Permutation importance | Global | Metric-aligned model reliance | Correlated substitutes and unrealistic permutations |
| PDP | Global response | Simple average effect curve | Extrapolation and hidden heterogeneity |
| ICE | Local-to-global response | Shows individual curves and interactions | Many curves and the same extrapolation risk as PDP |
| ALE | Global response | Uses local changes in observed regions | Sensitive to binning and sparse support |


#### **LIME and SHAP**

**LIME** explains a prediction by sampling around an instance $x_0$, querying the black-box model, weighting samples by proximity, and fitting an interpretable local surrogate:

$$
g^*
=
\arg\min_{g\in G}
\sum_{z\in\mathcal Z}
\pi_{x_0}(z)\bigl(f(z)-g(z)\bigr)^2
+\Omega(g).
$$

$\pi_{x_0}(z)$ defines locality and $\Omega(g)$ penalizes complexity. The explanation is the fitted surrogate $g$, not the original model. Its validity is restricted to the sampled neighborhood. Kernel width, perturbation distribution, feature discretization, sparsity, and random seed can materially change the coefficients.

A local explanation should report at least:

- the model output and output scale being approximated;
- the neighborhood generator and distance representation;
- the kernel width and effective sample size;
- local fidelity on weighted perturbations;
- stability across seeds and plausible neighborhood choices.

<details>
<summary><strong>Python: build and audit a LIME-style local surrogate</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Ridge

rng = np.random.default_rng(23)
X, y = make_moons(n_samples=1800, noise=0.22, random_state=23)
black_box = RandomForestClassifier(
    n_estimators=180, min_samples_leaf=6, random_state=23
).fit(X, y)

x0 = X[np.argmin(np.abs(black_box.predict_proba(X)[:, 1] - 0.55))]

def local_surrogate(kernel_width, seed):
    local_rng = np.random.default_rng(seed)
    perturbations = x0 + local_rng.normal(scale=0.35, size=(2500, 2))
    black_box_scores = black_box.predict_proba(perturbations)[:, 1]
    squared_distance = np.sum((perturbations - x0) ** 2, axis=1)
    weights = np.exp(-squared_distance / (2 * kernel_width**2))

    surrogate = Ridge(alpha=0.05).fit(
        perturbations, black_box_scores, sample_weight=weights
    )
    fitted = surrogate.predict(perturbations)
    weighted_mean = np.average(black_box_scores, weights=weights)
    weighted_sse = np.sum(weights * (black_box_scores - fitted) ** 2)
    weighted_sst = np.sum(weights * (black_box_scores - weighted_mean) ** 2)
    fidelity_r2 = 1 - weighted_sse / weighted_sst
    effective_n = weights.sum() ** 2 / np.sum(weights**2)
    return surrogate.coef_, fidelity_r2, effective_n

print("instance:", np.round(x0, 3))
print("black-box probability:", round(black_box.predict_proba(x0.reshape(1, -1))[0, 1], 3))
for width in [0.15, 0.30, 0.60]:
    coefficient_runs = []
    fidelities = []
    for seed in range(5):
        coefficients, fidelity, effective_n = local_surrogate(width, seed)
        coefficient_runs.append(coefficients)
        fidelities.append(fidelity)
    print(
        f"width={width:.2f}",
        "mean_coef=", np.round(np.mean(coefficient_runs, axis=0), 3),
        "coef_sd=", np.round(np.std(coefficient_runs, axis=0), 3),
        "mean_fidelity=", round(np.mean(fidelities), 3),
        "effective_n=", round(effective_n),
    )
```

</details>

High local fidelity does not imply that the neighborhood is realistic or relevant to the decision. Low fidelity means the proposed linear explanation is not adequate even on its own sampled region.

**SHAP** connects feature attribution to Shapley values from cooperative game theory. Define a value function $v(S)$ for a set of present features $S$. The contribution of feature $j$ is

$$
\phi_j
=
\sum_{S\subseteq F\setminus\{j\}}
\frac{|S|!(M-|S|-1)!}{M!}
\left[v(S\cup\{j\})-v(S)\right].
$$

The weighting averages feature $j$'s marginal contribution over every order in which features could enter. Under the chosen value function, Shapley values satisfy:

- **local accuracy or efficiency:** $f(x)=\phi_0+\sum_j\phi_j$;
- **missingness or null player:** a feature that never changes value receives zero;
- **symmetry:** interchangeable features receive equal credit;
- **additivity:** explanations add when games add.

The difficult step is not the combinatorics but defining “feature absent.” A common interventional value function replaces absent features with rows from a background dataset:

$$
v(S)=\mathbb E_{X_{\bar S}}\left[f(x_S,X_{\bar S})\right].
$$

This can break dependence between present and absent features. Conditional SHAP instead averages under $P(X_{\bar S}\mid X_S=x_S)$, preserving dependence but attributing shared information differently and requiring a conditional distribution. Neither is universally correct; each answers a different question.

<div class="diagram-scroll">

![An official SHAP waterfall plot moves from the background expectation to one model output through feature contributions.](assets/shap-waterfall-official.png){fig-alt="A SHAP waterfall chart starts at the expected model output and shows positive and negative feature contributions leading to one prediction."}

</div>

*Image source: [SHAP official waterfall example](https://shap.readthedocs.io/en/latest/example_notebooks/api_examples/plots/waterfall.html), from the [MIT-licensed SHAP project](https://github.com/shap/shap/blob/master/LICENSE). The displayed units are model-output units; they are not automatically probability points.*

<details>
<summary><strong>Python: compute exact interventional Shapley values for three features</strong></summary>

```python
import itertools
import math
import numpy as np

rng = np.random.default_rng(31)
background = rng.normal(size=(800, 3))
instance = np.array([1.2, -0.7, 0.9])

def model(matrix):
    # Deliberately nonlinear, with an interaction between features 0 and 1.
    return (
        0.8 * matrix[:, 0]
        - 0.5 * matrix[:, 1]
        + 1.1 * matrix[:, 0] * matrix[:, 1]
        + np.sin(matrix[:, 2])
    )

features = range(3)

def coalition_value(coalition):
    completed = background.copy()
    for feature in coalition:
        completed[:, feature] = instance[feature]
    return model(completed).mean()

values = {}
for size in range(4):
    for subset in itertools.combinations(features, size):
        values[frozenset(subset)] = coalition_value(subset)

shapley = np.zeros(3)
for feature in features:
    others = [j for j in features if j != feature]
    for size in range(3):
        for subset_tuple in itertools.combinations(others, size):
            subset = frozenset(subset_tuple)
            weight = (
                math.factorial(size)
                * math.factorial(3 - size - 1)
                / math.factorial(3)
            )
            shapley[feature] += weight * (
                values[subset | {feature}] - values[subset]
            )

baseline = values[frozenset()]
prediction = model(instance.reshape(1, -1))[0]
print("background value:", round(baseline, 4))
print("Shapley values:", np.round(shapley, 4))
print("baseline + contributions:", round(baseline + shapley.sum(), 4))
print("model prediction:", round(prediction, 4))
```

</details>

The efficiency check verifies the decomposition, but not whether the background population or missing-feature semantics are appropriate. SHAP values also explain the model output, including spurious or unfair behavior; they do not certify that the model is reasonable.

**LIME versus SHAP.**

| Question | LIME-style surrogate | SHAP-style attribution |
|---|---|---|
| Core object | Locally weighted interpretable model | Additive feature-credit allocation |
| Main design choice | Neighborhood and kernel | Value function and background distribution |
| Guarantee | Approximation fidelity is empirical | Shapley axioms under the chosen game |
| Common instability | Sampling and kernel width | Correlation and background choice |
| Best use | Inspect local shape with a simple surrogate | Decompose a prediction on a stated output scale |

#### **Counterfactual Explanations**

A counterfactual explanation searches for a feasible input $x'$ whose prediction satisfies a desired condition while remaining close to $x$:

$$
\min_{x'}
d(x,x')
+\lambda\,\ell\bigl(f(x'),y_{\text{target}}\bigr)
\quad\text{subject to}\quad
x'\in\mathcal F.
$$

$d$ encodes the cost of change and $\mathcal F$ encodes feasibility. Without constraints, the nearest mathematical counterfactual may reduce age, change a historical event, alter a protected characteristic, produce an impossible category combination, or exploit a model error. A useful system distinguishes:

- **immutable features:** age at decision, origin, or historical events;
- **mutable but not directly actionable features:** credit score or diagnosis;
- **actionable features:** a controllable payment, document, or configuration;
- **directional constraints:** debt may be reduced but not made negative;
- **causal constraints:** changing one variable updates its downstream consequences;
- **diversity:** several qualitatively different feasible options may be more useful than one optimum.

Counterfactual explanation is not automatically **recourse**. Recourse requires that the person can carry out the action, that the institution will apply the same policy later, and that the action is likely to produce the intended real-world outcome. A model counterfactual only states that $f(x')$ changes.

<details>
<summary><strong>Python: search for a constrained and auditable counterfactual</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(41)
n = 5000
income = rng.normal(65, 18, size=n)       # thousands per year
debt = np.clip(rng.normal(24, 12, size=n), 0, None)
age = rng.integers(21, 66, size=n)
X = np.column_stack([income, debt, age])

latent_score = 0.065 * income - 0.095 * debt + 0.012 * age - 3.2
y = rng.binomial(1, 1 / (1 + np.exp(-latent_score)))

scaler = StandardScaler().fit(X)
model = LogisticRegression(max_iter=2000).fit(scaler.transform(X), y)

# Select a person just below the operational threshold.
probabilities = model.predict_proba(scaler.transform(X))[:, 1]
eligible = np.flatnonzero((probabilities > 0.30) & (probabilities < 0.45))
index = eligible[np.argmax(probabilities[eligible])]
x = X[index].copy()
starting_probability = probabilities[index]

candidates = []
for income_increase in np.linspace(0, 30, 31):
    for debt_reduction in np.linspace(0, min(25, x[1]), 26):
        candidate = x.copy()
        candidate[0] += income_increase       # income can only increase
        candidate[1] -= debt_reduction        # debt can only decrease
        candidate[2] = x[2]                   # age is immutable
        probability = model.predict_proba(
            scaler.transform(candidate.reshape(1, -1))
        )[0, 1]
        if probability >= 0.50:
            # Domain-specific cost: one unit of debt reduction is treated as
            # more burdensome than one unit of additional annual income.
            cost = income_increase / 10 + debt_reduction / 5
            candidates.append((cost, probability, candidate))

cost, new_probability, counterfactual = min(candidates, key=lambda row: row[0])
print("original [income, debt, age]:", np.round(x, 2))
print("original probability:", round(starting_probability, 3))
print("counterfactual:", np.round(counterfactual, 2))
print("counterfactual probability:", round(new_probability, 3))
print("action cost:", round(cost, 3))
```

</details>

The optimization is only as defensible as its constraints and cost function. Counterfactuals should be stress-tested across model versions, checked for subgroup disparities in feasibility and cost, and separated from any claim that the recommended action causes the desired outcome.

**Summary.** LIME approximates local shape, SHAP allocates prediction credit under a chosen coalition game, and counterfactuals search for decision-changing inputs under constraints. None of them independently proves causal recourse, fairness, or model correctness.


### **Robustness and Distribution Shift**

Robustness is the ability of a system to preserve acceptable behavior under a specified range of departures from its nominal conditions. The word is incomplete without three elements:

1. a **perturbation or shift set** describing what can change;
2. a **performance requirement** describing what must remain acceptable;
3. an **operating response** describing detection, abstention, fallback, recovery, and monitoring.

A model can be robust to Gaussian sensor noise and brittle to missing fields; robust within one hospital and unsafe in another; resistant to small image perturbations and vulnerable to data poisoning. Robustness is always relative to a threat or environment model.

#### **Noise, Corruption, and Stress Testing**

Let $P_{\text{train}}(X,Y)$ be the data-generating distribution represented by training data and $P_{\text{deploy}}(X,Y)$ the deployment distribution. Common shifts include:

- **Covariate shift:** $P(X)$ changes while $P(Y\mid X)$ is assumed stable.
- **Label shift:** $P(Y)$ changes while $P(X\mid Y)$ is assumed stable.
- **Concept shift:** $P(Y\mid X)$ changes, so the predictive relationship itself is no longer stable.
- **Subgroup or mixture shift:** the prevalence of environments or groups changes, or performance changes within one group.
- **Support shift:** deployment contains regions with little or no training support.
- **Measurement shift:** the real construct is similar, but sensors, coding rules, missingness, or preprocessing differ.

<div class="diagram-scroll">

![Different distribution shifts change different components of the data-generating process.](assets/distribution-shift-taxonomy.svg){fig-alt="Four panels compare covariate, label, concept, and subgroup shift and show that each requires different assumptions and responses."}

</div>

These labels are not directly observable facts. Calling a problem covariate shift asserts that the conditional mechanism remained stable. That assumption should be defended with domain knowledge and tested where labels eventually arrive.

A **stress test** applies plausible corruptions over a severity range and measures more than average accuracy:

- task loss, calibration, and abstention rate;
- worst-group and lower-quantile performance;
- sensitivity to missingness patterns and schema changes;
- monotonic degradation rather than one arbitrary corruption level;
- confidence intervals and repeated corruption draws;
- operational latency, resource exhaustion, and fallback success.

Corruptions should reflect the deployment process. Gaussian noise is useful for a unit test but rarely represents all real failures. For tabular data, test clipping, rounding, unit mistakes, stale fields, sentinel values, missing-not-at-random patterns, category drift, and duplicated records. For text, test typos, code switching, negation, dialect, prompt injection, and truncation. For images and audio, test blur, compression, lighting, occlusion, device changes, and temporal artifacts.

<details>
<summary><strong>Python: build a severity-based stress matrix with subgroup reporting</strong></summary>

```python
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(53)
n = 6000
X = rng.normal(size=(n, 8))
true_logit = (
    1.45 * X[:, 0]
    - 1.20 * X[:, 1]
    + 1.05 * X[:, 2]
    + 0.75 * X[:, 3]
    - 0.35 * X[:, 4]
)
true_probability = 1 / (1 + np.exp(-true_logit))
y = rng.binomial(1, true_probability)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=53
)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=2000),
).fit(X_train, y_train)

# This is an audit slice, not a claim that the feature defines a social group.
group = X_test[:, 1] > np.median(X_train[:, 1])

def evaluate(name, changed):
    probability = model.predict_proba(changed)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    group_accuracies = [
        accuracy_score(y_test[group == value], prediction[group == value])
        for value in [False, True]
    ]
    return {
        "scenario": name,
        "accuracy": accuracy_score(y_test, prediction),
        "log_loss": log_loss(y_test, probability),
        "worst_group_accuracy": min(group_accuracies),
        "group_gap": abs(group_accuracies[0] - group_accuracies[1]),
    }

results = [evaluate("clean", X_test)]
for severity in [0.15, 0.35, 0.70]:
    noisy = X_test + rng.normal(scale=severity, size=X_test.shape)
    results.append(evaluate(f"noise-{severity:.2f}", noisy))

for missing_rate in [0.05, 0.15, 0.30]:
    missing = X_test.copy()
    mask = rng.random(missing.shape) < missing_rate
    missing[mask] = np.nan
    results.append(evaluate(f"missing-{missing_rate:.2f}", missing))

for row in results:
    print(
        f"{row['scenario']:14s}",
        f"acc={row['accuracy']:.3f}",
        f"logloss={row['log_loss']:.3f}",
        f"worst_group={row['worst_group_accuracy']:.3f}",
        f"gap={row['group_gap']:.3f}",
    )
```

</details>

The worst-group column can deteriorate faster than the aggregate. A deployment gate should be tied to explicit tolerances, such as “worst-slice recall remains above 0.82 through the expected 95th percentile of missingness,” rather than a vague statement that degradation is “small.”

Robustness interventions operate at several levels:

| Level | Examples | Limitation |
|---|---|---|
| Data | Better coverage, augmentation, corruption simulation, reweighting | Synthetic perturbations may not match deployment |
| Model | Regularization, invariant features, robust loss, adversarial training | Can trade clean utility for a narrow robustness target |
| Uncertainty | Calibration, ensembles, conformal sets, OOD scores | Uncertainty can also fail under shift |
| System | Validation, schema contracts, fallback, human escalation, rollback | Requires operational ownership and testing |
| Monitoring | Drift, slice metrics, delayed labels, incident review | Detection is not correction |

#### **Adversarial Examples**

An **adversarial example** is an input deliberately modified to cause a model error while satisfying a specified perturbation constraint. For a classifier with parameters $\theta$, loss $\ell$, input $x$, and label $y$, an untargeted evasion attack solves

$$
\max_{\delta\in\Delta}
\ell\bigl(f_\theta(x+\delta),y\bigr),
$$

where $\Delta$ may constrain $\|\delta\|_\infty\le\epsilon$, preserve semantic validity, restrict editable fields, or model physical transformations. A targeted attack instead pushes the output toward an attacker-chosen class.

The **Fast Gradient Sign Method (FGSM)** takes one step:

$$
x_{\text{adv}}
=
\operatorname{clip}\left(
x+\epsilon\,\operatorname{sign}
\left(\nabla_x\ell(f_\theta(x),y)\right)
\right).
$$

Projected gradient descent repeats smaller steps and projects back into $\Delta$. These formulas are meaningful only after a threat model states:

- attacker goal: evasion, poisoning, extraction, or inference;
- capability: which inputs, training points, labels, or queries can be changed;
- knowledge: white-box parameters, gradients, architecture, data, or black-box outputs;
- budget: norm, number of fields, semantic constraint, query count, or physical cost;
- success criterion and defensive assumptions.

<div class="diagram-scroll">

![The Adversarial Robustness Toolbox distinguishes evasion, poisoning, extraction, and inference threats.](assets/art-adversarial-threats-official.png){fig-alt="A four-part puzzle graphic labels evasion, poisoning, extraction, and inference as adversarial machine learning threat categories."}

</div>

*Image source: [Adversarial Robustness Toolbox threat graphic](https://github.com/Trusted-AI/adversarial-robustness-toolbox/blob/main/docs/images/adversarial_threats_art.png), [MIT license](https://github.com/Trusted-AI/adversarial-robustness-toolbox/blob/main/LICENSE).*

<details>
<summary><strong>Python: implement FGSM against a differentiable logistic classifier</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=5000,
    n_features=12,
    n_informative=8,
    class_sep=1.3,
    random_state=61,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=61
)

scaler = StandardScaler().fit(X_train)
X_train_z = scaler.transform(X_train)
X_test_z = scaler.transform(X_test)
model = LogisticRegression(max_iter=2000).fit(X_train_z, y_train)

weight = model.coef_[0]
bias = model.intercept_[0]

def sigmoid(value):
    value = np.clip(value, -40, 40)
    return 1 / (1 + np.exp(-value))

def predict(matrix):
    return (sigmoid(matrix @ weight + bias) >= 0.5).astype(int)

def fgsm(matrix, labels, epsilon):
    probability = sigmoid(matrix @ weight + bias)
    # Gradient of binary cross-entropy with respect to standardized input.
    gradient = (probability - labels)[:, None] * weight[None, :]
    return matrix + epsilon * np.sign(gradient)

print("clean accuracy:", round(accuracy_score(y_test, predict(X_test_z)), 3))
for epsilon in [0.02, 0.05, 0.10, 0.20]:
    adversarial = fgsm(X_test_z, y_test, epsilon)
    linf = np.max(np.abs(adversarial - X_test_z), axis=1).mean()
    print(
        f"epsilon={epsilon:.2f}",
        f"mean_Linf={linf:.3f}",
        f"accuracy={accuracy_score(y_test, predict(adversarial)):.3f}",
    )
```

</details>

This attack is exactly aligned with the standardized logistic model, so it is a transparent demonstration rather than a realistic security certification. For categorical or constrained tabular data, an $L_p$ ball can permit impossible records. For text, synonymous substitutions can alter semantics; for images, pixel norms do not guarantee perceptual equivalence. Evaluate attacks that match the actual interface and compare against adaptive attackers who know the defense.

Adversarial training approximately solves

$$
\min_\theta
\mathbb E_{(X,Y)}
\left[
\max_{\delta\in\Delta}
\ell(f_\theta(X+\delta),Y)
\right].
$$

It can improve performance inside the trained threat set, but robustness may not transfer to a different norm, budget, attack, corruption, or distribution. Gradient masking can make weak attacks appear unsuccessful without making the model secure; use multiple strong attacks, attack restarts, sanity checks, and independent evaluation.

#### **Out-of-Distribution Detection**

An out-of-distribution (OOD) detector attempts to identify inputs outside the distribution for which model behavior was validated. This requires defining “outside” relative to a training or deployment reference. Near-OOD examples share low-level structure but differ in meaningful classes or environments; far-OOD examples can be visually or statistically obvious.

Common scores include:

- maximum softmax or class probability;
- predictive entropy or ensemble disagreement;
- distance in input or representation space;
- density or likelihood under a generative model;
- energy scores, one-class models, and reconstruction error.

No score is universally reliable. A discriminative classifier can be highly confident far from training support, and a likelihood model can assign high density to irrelevant but simple inputs. Detection should be evaluated on several realistic OOD sources, with AUROC, area under the precision-recall curve, false-positive rate at a target true-positive rate, and operating-threshold uncertainty.

OOD detection is often paired with **selective prediction**. Let the model predict only when a confidence score $s(x)$ exceeds threshold $\tau$:

$$
\operatorname{coverage}(\tau)
=P(s(X)\ge\tau),
$$

$$
\operatorname{risk}(\tau)
=
\mathbb E\!\left[
L(Y,f(X))
\mid s(X)\ge\tau
\right].
$$

The risk-coverage curve shows how error among accepted cases changes as the system abstains more often.

<div class="diagram-scroll">

![A risk-coverage curve makes the abstention trade-off explicit.](assets/risk-coverage-curve.svg){fig-alt="A curve shows conditional risk increasing with coverage, with a marked threshold that balances accepted cases and abstentions."}

</div>

<details>
<summary><strong>Python: compare an OOD score and construct a risk-coverage table</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(71)

# Two in-distribution classes.
class_0 = rng.normal(loc=[-1.2, 0.0], scale=[0.75, 0.85], size=(1500, 2))
class_1 = rng.normal(loc=[1.2, 0.0], scale=[0.75, 0.85], size=(1500, 2))
X_train = np.vstack([class_0[:1000], class_1[:1000]])
y_train = np.r_[np.zeros(1000, dtype=int), np.ones(1000, dtype=int)]
X_test = np.vstack([class_0[1000:], class_1[1000:]])
y_test = np.r_[np.zeros(500, dtype=int), np.ones(500, dtype=int)]

# OOD data lie above both classes but can still receive confident class scores.
X_ood = rng.normal(loc=[0.0, 4.0], scale=[1.1, 0.65], size=(1000, 2))

scaler = StandardScaler().fit(X_train)
train_z = scaler.transform(X_train)
test_z = scaler.transform(X_test)
ood_z = scaler.transform(X_ood)

model = LogisticRegression(max_iter=2000).fit(train_z, y_train)
test_probability = model.predict_proba(test_z)
ood_probability = model.predict_proba(ood_z)

confidence_test = test_probability.max(axis=1)
confidence_ood = ood_probability.max(axis=1)
confidence_ood_score = 1 - np.r_[confidence_test, confidence_ood]

neighbors = NearestNeighbors(n_neighbors=10).fit(train_z)
distance_test = neighbors.kneighbors(test_z, return_distance=True)[0].mean(axis=1)
distance_ood = neighbors.kneighbors(ood_z, return_distance=True)[0].mean(axis=1)
distance_ood_score = np.r_[distance_test, distance_ood]

ood_label = np.r_[np.zeros(len(test_z)), np.ones(len(ood_z))]
print("OOD AUROC from low confidence:", round(roc_auc_score(ood_label, confidence_ood_score), 3))
print("OOD AUROC from neighbor distance:", round(roc_auc_score(ood_label, distance_ood_score), 3))

# Risk-coverage on labeled in-distribution test data.
prediction = test_probability.argmax(axis=1)
error = (prediction != y_test).astype(float)
order = np.argsort(-confidence_test)
for coverage in [0.25, 0.50, 0.75, 1.00]:
    accepted = order[: max(1, int(coverage * len(order)))]
    threshold = confidence_test[accepted].min()
    print(
        f"coverage={coverage:.2f}",
        f"threshold={threshold:.3f}",
        f"selective_risk={error[accepted].mean():.3f}",
    )
```

</details>

Distance works well in this deliberately geometric example, but can become meaningless in a high-dimensional raw space. Representation choice is part of the detector, and the threshold should be selected using deployment costs: abstaining on too many cases can overload human review or systematically deny service.

**Summary.** Robustness requires a named shift or adversary, a severity range, slice-aware metrics, and a tested response. Stress testing, adversarial evaluation, OOD detection, and abstention are complementary. None replaces monitoring with delayed ground-truth labels, because the most important concept changes may not be visible from $X$ alone.


### **Fairness in Machine Learning**

Fairness is not a statistical property that can be selected independently of the application. It is a normative and sociotechnical question about **which people face which harms, under which decision process, with what opportunity for review or remedy**. Quantitative metrics help test a stated concern; they do not decide which concern matters.

A fairness analysis should specify:

1. the decision, score, ranking, or allocation being evaluated;
2. affected people, relevant groups, and intersectional groups;
3. the benefit or harm, including false positives, false negatives, delay, quality of service, and denial of opportunity;
4. the time horizon and feedback loop;
5. the fairness criterion and why it matches that harm;
6. uncertainty, sample support, and practically meaningful tolerances;
7. who owns the mitigation decision and how affected people can contest it.

#### **Sources of Bias**

Bias can enter before, during, and after model fitting:

| Stage | Mechanism | Example diagnostic |
|---|---|---|
| Problem formulation | The target or decision encodes an unjust policy | Compare the stated objective with the actual institutional goal |
| Sampling | Some populations are missing or selectively observed | Coverage, nonresponse, and support by group |
| Measurement | Features or labels measure groups with different error | Inter-rater agreement and label validity by group |
| Historical process | Labels reproduce past allocation or enforcement | Audit how labels were generated and who was exposed |
| Representation | Features contain proxies or lose relevant context | Conditional error and representation distances |
| Learning | Average loss sacrifices a smaller or harder group | Per-group learning curves and worst-group risk |
| Thresholding | One score threshold creates different consequences | Confusion rates and utility by group |
| Deployment | Users adapt, automation changes behavior, appeals differ | Process metrics, overrides, delays, and longitudinal outcomes |

Removing a protected attribute does not guarantee fairness. Other variables can proxy it, and the attribute may be needed to measure disparities, learn group-specific measurement error, or implement a legally and ethically justified mitigation. Conversely, using an attribute can create new risks. Access, purpose, and governance must be explicit.

Fairness should be measured on the **decision population**, not only the rows for which labels happen to be available. Selective labels are common: loan repayment is observed only for approved applicants, treatment outcomes only for treated patients, and reoffending may be measured through unequal surveillance. Standard test metrics can then condition on a biased observation process.

#### **Demographic Parity and Equalized Odds**

Let protected or audit group be $A$, true label $Y$, prediction $\hat Y$, and score $S$.

**Demographic parity** requires equal positive decision rates:

$$
P(\hat Y=1\mid A=a)
=
P(\hat Y=1\mid A=b).
$$

It is relevant when the allocation rate itself is the concern or when labels are not trustworthy, but it ignores qualification labels and can require very different error rates when base rates differ.

**Equal opportunity** requires equal true-positive rates:

$$
P(\hat Y=1\mid Y=1,A=a)
=
P(\hat Y=1\mid Y=1,A=b).
$$

It focuses on access among label-positive cases. **Equalized odds** additionally requires equal false-positive rates:

$$
\hat Y\perp A\mid Y.
$$

Equivalently, both

$$
TPR_a=TPR_b
\quad\text{and}\quad
FPR_a=FPR_b.
$$

Which error matters depends on the application. In a safety alert, a false negative can miss danger; in an accusation system, a false positive can impose serious harm. Equalized odds treats both error-rate differences as relevant but does not encode their relative cost.

**Predictive parity** requires equal positive predictive value:

$$
P(Y=1\mid \hat Y=1,A=a)
=
P(Y=1\mid \hat Y=1,A=b).
$$

This asks whether a positive decision has the same meaning across groups. It is distinct from equalized odds, which conditions on the true label.

<div class="diagram-scroll">

![Common fairness criteria impose different conditional independence relationships.](assets/fairness-criteria-map.svg){fig-alt="Four panels define demographic parity, equalized odds, predictive parity, and calibration, followed by a warning that they generally conflict when base rates differ."}

</div>

<details>
<summary><strong>Python: audit selection, error, and predictive-value disparities</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(83)
n = 12_000
group = rng.binomial(1, 0.45, size=n)
x = rng.normal(loc=0.25 * group, scale=1.0, size=n)

# Different group base rates and measurement quality produce several kinds
# of disparity even though the same threshold is used.
true_logit = -0.8 + 1.25 * x + 0.65 * group
true_probability = 1 / (1 + np.exp(-true_logit))
y = rng.binomial(1, true_probability)

score_logit = -0.65 + 1.05 * x + 0.15 * group + rng.normal(
    scale=0.45 + 0.25 * group, size=n
)
score = 1 / (1 + np.exp(-score_logit))
prediction = (score >= 0.50).astype(int)

def safe_ratio(numerator, denominator):
    return numerator / denominator if denominator else np.nan

def group_metrics(group_value):
    mask = group == group_value
    truth = y[mask]
    pred = prediction[mask]
    tp = np.sum((truth == 1) & (pred == 1))
    fp = np.sum((truth == 0) & (pred == 1))
    tn = np.sum((truth == 0) & (pred == 0))
    fn = np.sum((truth == 1) & (pred == 0))
    return {
        "n": mask.sum(),
        "base_rate": truth.mean(),
        "selection_rate": pred.mean(),
        "tpr": safe_ratio(tp, tp + fn),
        "fpr": safe_ratio(fp, fp + tn),
        "ppv": safe_ratio(tp, tp + fp),
    }

metrics = [group_metrics(value) for value in [0, 1]]
for value, row in enumerate(metrics):
    display = {
        key: int(val) if key == "n" else round(float(val), 3)
        for key, val in row.items()
    }
    print("group", value, display)

print("\nabsolute gaps")
for key in ["selection_rate", "tpr", "fpr", "ppv"]:
    print(key, round(abs(metrics[0][key] - metrics[1][key]), 3))
```

</details>

Report denominators as well as rates. A 10-point false-positive gap supported by 20 negative examples has very different uncertainty from the same gap supported by 20,000. For ranking systems, examine exposure and position-weighted utility rather than forcing every output into binary classification.

#### **Calibration and Individual Fairness**

A probabilistic score is **calibrated within groups** when

$$
P(Y=1\mid S=s,A=a)=s
$$

for relevant score regions and every group $a$. In practice, calibration is estimated in bins or with smooth calibration curves and accompanied by uncertainty. Average calibration error can hide severe local errors, especially in sparse high-risk regions.

When groups have different base rates and prediction is imperfect, several desirable criteria are generally incompatible. A score can be calibrated in every group yet produce different false-positive and true-positive rates under a common threshold. Enforcing equalized odds can require group-dependent randomized decisions that no longer preserve the same score semantics. This is not a software bug; it means the criteria encode different ideas of fairness.

<details>
<summary><strong>Python: show calibrated group scores that do not satisfy equalized odds</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(89)
n = 100_000
group = rng.binomial(1, 0.5, size=n)

# Scores come from different beta distributions, so group base rates differ.
# Drawing Y ~ Bernoulli(score) makes the score calibrated by construction.
score = np.empty(n)
score[group == 0] = rng.beta(2.0, 5.0, size=np.sum(group == 0))
score[group == 1] = rng.beta(5.0, 2.5, size=np.sum(group == 1))
y = rng.binomial(1, score)
prediction = score >= 0.5

def expected_calibration_error(mask, bins=12):
    edges = np.linspace(0, 1, bins + 1)
    total = mask.sum()
    ece = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        in_bin = mask & (score >= left) & (
            (score < right) if right < 1 else (score <= right)
        )
        if in_bin.any():
            ece += in_bin.sum() / total * abs(y[in_bin].mean() - score[in_bin].mean())
    return ece

for value in [0, 1]:
    mask = group == value
    positives = mask & (y == 1)
    negatives = mask & (y == 0)
    tpr = prediction[positives].mean()
    fpr = prediction[negatives].mean()
    ppv = y[mask & prediction].mean()
    print(
        f"group={value}",
        f"base_rate={y[mask].mean():.3f}",
        f"ECE={expected_calibration_error(mask):.4f}",
        f"TPR={tpr:.3f}",
        f"FPR={fpr:.3f}",
        f"PPV={ppv:.3f}",
    )
```

</details>

The near-zero calibration errors coexist with different TPR, FPR, and PPV. Choosing a metric therefore requires a substantive argument about harm, label validity, and decision meaning, not metric shopping.

**Individual fairness** is often expressed as “similar individuals should be treated similarly”:

$$
d_Y\bigl(f(x_i),f(x_j)\bigr)
\le
L\,d_X(x_i,x_j).
$$

The mathematical inequality is straightforward; defining a fair task-specific distance $d_X$ is not. Historical features can encode injustice, and two people who look similar in recorded data may face different constraints. **Counterfactual fairness** instead asks whether a decision would remain unchanged across counterfactual worlds where a protected attribute differs while appropriate background factors are held fixed. It requires a causal model and inherits its untestable assumptions.

Group and individual fairness can conflict. A smooth, individually consistent rule can preserve a group disparity inherited from the data, while a group-parity intervention can assign different decisions to otherwise similar people near a threshold. The conflict should be documented rather than hidden behind a composite score.

#### **Pre-, In-, and Post-Processing Mitigation**

Fairness mitigation can modify data, learning, or decisions.

<div class="diagram-scroll">

![Fairness mitigation can operate before, during, or after model training.](assets/fairness-mitigation-pipeline.svg){fig-alt="A three-stage pipeline compares preprocessing, in-processing, and post-processing fairness interventions and their limitations."}

</div>

**Pre-processing** methods reweight, resample, relabel, or transform data. They can work with an unchanged learner but may conceal the intervention from downstream users and cannot repair an invalid target. **In-processing** methods add fairness constraints, robust group objectives, or adversarial representation losses to training. They directly optimize the chosen criterion but require access to learning internals and careful generalization checks. **Post-processing** changes thresholds or randomizes decisions after fitting. It is convenient for an existing score, but can require group information at decision time and may reduce score coherence.

For group-dependent thresholds $t_a$, one constrained formulation is

$$
\min_{\{t_a\}}
\widehat R(\{t_a\})
\quad\text{subject to}\quad
\max_{a,b}
\left\{
|TPR_a-TPR_b|,
|FPR_a-FPR_b|
\right\}
\le \epsilon.
$$

$\epsilon$ is an explicit tolerance, not a magical definition of fairness. Thresholds should be fitted on validation data and evaluated once on untouched test data; optimizing and reporting on the same sample overstates parity.

<details>
<summary><strong>Python: search group thresholds and reveal the utility-parity trade-off</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(97)
n = 18_000
group = rng.binomial(1, 0.45, size=n)
ability = rng.normal(size=n)
true_probability = 1 / (1 + np.exp(-(-0.4 + 1.35 * ability + 0.45 * group)))
y = rng.binomial(1, true_probability)

# Group 1 receives a noisier score.
raw_score = -0.35 + 1.15 * ability + rng.normal(
    scale=0.55 + 0.45 * group, size=n
)
score = 1 / (1 + np.exp(-raw_score))

def evaluate(threshold_0, threshold_1):
    threshold = np.where(group == 0, threshold_0, threshold_1)
    prediction = score >= threshold
    tpr, fpr = [], []
    for value in [0, 1]:
        mask = group == value
        tpr.append(prediction[mask & (y == 1)].mean())
        fpr.append(prediction[mask & (y == 0)].mean())
    error = np.mean(prediction != y)
    equalized_odds_gap = max(abs(tpr[0] - tpr[1]), abs(fpr[0] - fpr[1]))
    return error, equalized_odds_gap, tpr, fpr

baseline = evaluate(0.5, 0.5)
candidate_rows = []
for threshold_0 in np.linspace(0.20, 0.80, 31):
    for threshold_1 in np.linspace(0.20, 0.80, 31):
        metrics = evaluate(threshold_0, threshold_1)
        candidate_rows.append((metrics[0] + 1.2 * metrics[1], threshold_0, threshold_1, metrics))

_, best_t0, best_t1, mitigated = min(candidate_rows, key=lambda row: row[0])
print(
    "shared threshold:",
    f"error={baseline[0]:.3f}",
    f"EO_gap={baseline[1]:.3f}",
    "TPR=", np.round(baseline[2], 3),
    "FPR=", np.round(baseline[3], 3),
)
print(
    f"group thresholds: t0={best_t0:.2f}, t1={best_t1:.2f}",
    f"error={mitigated[0]:.3f}",
    f"EO_gap={mitigated[1]:.3f}",
    "TPR=", np.round(mitigated[2], 3),
    "FPR=", np.round(mitigated[3], 3),
)
```

</details>

The penalty weight selects one point on a utility-parity frontier. A different social cost, tolerance, or group definition selects another point. Report the frontier and the decision rule rather than presenting the selected model as objectively “debiased.”

Fairness generalization is especially fragile for small and intersectional groups. A metric that is close on the full test set can vary substantially across bootstrap samples or fail within the intersection of age, gender, location, disability, language, or device access.

<details>
<summary><strong>Python: quantify uncertainty for intersectional recall</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(101)
n = 9000
group_a = rng.binomial(1, 0.35, size=n)
group_b = rng.binomial(1, 0.25, size=n)
signal = rng.normal(size=n)

true_probability = 1 / (1 + np.exp(-(-0.5 + 1.25 * signal)))
y = rng.binomial(1, true_probability)

# The small intersection receives both noisier and systematically lower scores.
intersection = (group_a == 1) & (group_b == 1)
noise_scale = np.where(intersection, 1.35, 0.55)
score = 1 / (
    1
    + np.exp(
        -(
            -0.45
            + signal
            - 0.85 * intersection
            + rng.normal(scale=noise_scale)
        )
    )
)
prediction = score >= 0.5

def recall(indices):
    positives = indices[y[indices] == 1]
    return prediction[positives].mean() if len(positives) else np.nan

for a in [0, 1]:
    for b in [0, 1]:
        indices = np.flatnonzero((group_a == a) & (group_b == b))
        point = recall(indices)
        bootstrap = []
        for _ in range(500):
            sample = rng.choice(indices, size=len(indices), replace=True)
            value = recall(sample)
            if not np.isnan(value):
                bootstrap.append(value)
        lower, upper = np.quantile(bootstrap, [0.025, 0.975])
        print(
            f"A={a}, B={b}",
            f"n={len(indices):4d}",
            f"recall={point:.3f}",
            f"95% bootstrap interval=({lower:.3f}, {upper:.3f})",
        )
```

</details>

Multiple comparisons also matter: searching hundreds of slices and reporting only the largest gap will find noise. Pre-specify critical slices, control false discoveries for exploratory searches, and investigate persistent disparities with domain experts and affected communities.

**Summary.** A fairness metric is a test of a stated harm model, not a universal certificate. Report base rates, confusion components, sample sizes, uncertainty, intersections, and utility. Evaluate the complete decision process, including who receives labels, who can appeal, and how the system changes future data.


### **Privacy**

Privacy risk arises when data about a person or organization can be learned beyond the intended purpose. Removing names is rarely sufficient. Quasi-identifiers can re-identify records, model parameters can memorize rare examples, gradients can reveal training content, and repeated queries can accumulate evidence.

A privacy analysis begins with a **threat model**:

- **Privacy unit:** one row, person, household, device, client, event, or all records belonging to one person.
- **Protected information:** participation, an attribute, raw content, relationship, model update, or population statistic.
- **Adversary:** model user, server, other client, insider, data recipient, or external observer.
- **Access:** labels, probabilities, embeddings, gradients, parameters, timing, or repeated adaptive queries.
- **Auxiliary knowledge:** public records, partial features, shadow data, or knowledge of the training algorithm.
- **Release history:** previous models, dashboards, checkpoints, and correlated datasets.

<div class="diagram-scroll">

![Privacy leakage can pass through data, model parameters, interfaces, and auxiliary information.](assets/privacy-threat-surface.svg){fig-alt="A pipeline from training data through a model and interface to an attacker lists membership inference, gradient leakage, adaptive queries, and auxiliary data."}

</div>

Data minimization, retention limits, encryption, authentication, access control, logging, and incident response remain necessary. A mathematical privacy guarantee controls a defined information channel; it does not repair an exposed database, malicious client, vulnerable endpoint, or side channel outside the model.

#### **Membership and Attribute Inference**

A **membership inference attack** predicts whether a target record participated in training. Overfit models often produce lower loss or higher confidence on training records, giving an attacker a signal. An attack can use the target label, probability vector, per-example loss, gradients, or repeated augmented queries.

Attack performance should be measured against a realistic membership prior and operating point. AUROC summarizes ranking but can hide a high false-positive rate in a population where only a tiny fraction are members. Report precision or advantage at deployment-relevant priors and include strong baselines.

<details>
<summary><strong>Python: turn a generalization gap into a membership signal</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(
    n_samples=4000,
    n_features=80,
    n_informative=16,
    n_redundant=8,
    flip_y=0.08,
    random_state=107,
)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, train_size=700, test_size=700, stratify=y, random_state=107
)

models = {
    "unpruned tree": DecisionTreeClassifier(random_state=107),
    "regularized logistic": LogisticRegression(C=0.15, max_iter=3000),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    train_probability = model.predict_proba(X_train)
    holdout_probability = model.predict_proba(X_holdout)

    # Attacker knows each target's true label and uses confidence assigned to it.
    train_confidence = train_probability[np.arange(len(y_train)), y_train]
    holdout_confidence = holdout_probability[np.arange(len(y_holdout)), y_holdout]
    attack_score = np.r_[train_confidence, holdout_confidence]
    membership = np.r_[np.ones(len(y_train)), np.zeros(len(y_holdout))]

    print(
        name,
        f"train_acc={accuracy_score(y_train, model.predict(X_train)):.3f}",
        f"test_acc={accuracy_score(y_holdout, model.predict(X_holdout)):.3f}",
        f"membership_AUROC={roc_auc_score(membership, attack_score):.3f}",
    )
```

</details>

Regularization and good generalization can reduce this simple attack but do not provide a worst-case privacy guarantee. Rare or duplicated records may remain vulnerable even when average train and test losses match.

An **attribute inference attack** uses observed attributes and model access to infer a hidden sensitive attribute. A **model inversion or reconstruction attack** attempts to recover representative or specific training content. **Model extraction** reconstructs model behavior or parameters through queries; it is often an intellectual-property or security concern and can enable stronger privacy attacks. These categories overlap, so evaluation should follow the actual adversary objective rather than rely only on a taxonomy label.

Defenses include output minimization, confidence rounding, rate limits, audit logging, regularization, early stopping, data deduplication, access control, private aggregation, and differential privacy. Output obfuscation can reduce one attack while preserving leakage under adaptive queries; it should be tested against an attacker aware of the defense.

#### **Differential Privacy**

Differential privacy (DP) bounds how much an algorithm's output distribution can change when one protected unit is added, removed, or replaced. A randomized mechanism $\mathcal M$ is $(\epsilon,\delta)$-differentially private if, for every pair of neighboring datasets $D\sim D'$ and measurable output set $S$,

$$
P\bigl(\mathcal M(D)\in S\bigr)
\le
e^\epsilon
P\bigl(\mathcal M(D')\in S\bigr)
+\delta.
$$

The definition is worst-case over neighboring datasets and outputs. Smaller $\epsilon$ gives a tighter multiplicative bound; $\delta$ allows a small additive failure probability and should be chosen relative to the number of protected units and the application. An $\epsilon$ value is uninterpretable without the neighboring relation, privacy unit, mechanism, composition history, and threat model.

For a numeric query $q$, the global $L_1$ sensitivity is

$$
\Delta_1 q
=
\max_{D\sim D'}
\|q(D)-q(D')\|_1.
$$

The Laplace mechanism releases

$$
\widetilde q(D)
=
q(D)+\operatorname{Laplace}
\left(0,\frac{\Delta_1q}{\epsilon}\right),
$$

which satisfies pure $\epsilon$-DP under the corresponding assumptions. The Gaussian mechanism is commonly used for approximate DP and must calibrate noise to sensitivity, $\epsilon$, and $\delta$.

<details>
<summary><strong>Python: connect sensitivity, epsilon, utility, and composition</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(113)
true_count = 320
sensitivity = 1.0  # add/remove-one neighboring relation for a count query
draws = 50_000

for epsilon in [0.10, 0.30, 1.00, 3.00]:
    scale = sensitivity / epsilon
    released = true_count + rng.laplace(loc=0.0, scale=scale, size=draws)
    absolute_error = np.abs(released - true_count)
    print(
        f"epsilon={epsilon:>4.2f}",
        f"noise_scale={scale:>5.2f}",
        f"median_abs_error={np.median(absolute_error):>5.2f}",
        f"95th_percentile_error={np.quantile(absolute_error, 0.95):>6.2f}",
    )

# Basic sequential composition: ten epsilon=0.3 releases about the same
# protected units provide an upper bound of epsilon_total=3.0.
epsilon_per_release = 0.3
number_of_releases = 10
print(
    "basic-composition epsilon:",
    epsilon_per_release * number_of_releases,
)
```

</details>

Clipping or bounding is essential because unbounded values can have unbounded sensitivity. For a mean, clip each contribution to a declared interval before adding noise and account for the bias this introduces. Privacy and statistical bias are separate concerns.

For machine learning, **DP-SGD** performs the following for each minibatch:

1. compute a gradient $g_i$ for each example or protected unit;
2. clip it to norm $C$:

   $$
   \bar g_i
   =
   g_i\min\left(1,\frac{C}{\|g_i\|_2}\right);
   $$

3. aggregate clipped gradients and add Gaussian noise;
4. update parameters;
5. use a privacy accountant to compose loss across sampling steps and releases.

Clipping bounds one unit's influence; noise obscures that bounded contribution. The clipping norm, noise multiplier, sampling scheme, number of steps, group privacy, checkpoints, hyperparameter search, and released metrics all affect the final guarantee. Privacy accounting must include everything derived from private data that is released, not only the final model.

<div class="diagram-scroll">

![NIST's differential privacy pyramid places epsilon above the privacy unit, algorithms, threat model, security, access control, and collection exposure.](assets/nist-differential-privacy-pyramid.png){fig-alt="A NIST pyramid shows epsilon at the top, unit of privacy and algorithm correctness in the middle, and threat model, security, access control, and data collection exposure at the base."}

</div>

*Image source and credit: [NIST Differential Privacy Pyramid](https://www.nist.gov/image/differential-privacy-pyramid). The diagram emphasizes that an epsilon claim depends on the implementation and operational layers beneath it.*

DP has important closure properties. **Post-processing** cannot worsen a valid DP guarantee if it uses no additional private information. **Composition** makes privacy loss accumulate across releases. **Group privacy** weakens as the number of linked records per protected person grows. These properties make DP auditable, but only when the implementation matches the analyzed mechanism.

#### **Federated Learning**

Federated learning trains a shared model while clients retain raw data locally. In synchronous **federated averaging**, client $k$ starts from global parameters $w_t$, performs local optimization to obtain $w_{t+1}^{(k)}$, and the server aggregates

$$
w_{t+1}
=
\sum_{k=1}^{K}
\frac{n_k}{\sum_jn_j}
w_{t+1}^{(k)}.
$$

This reduces centralized data movement, supports data residency, and can use distributed computation. It does **not** by itself provide confidentiality or differential privacy. Individual updates can leak information, a malicious server can manipulate model states, malicious clients can poison training, and the final model can memorize records.

<div class="diagram-scroll">

![Federated learning, secure aggregation, and differential privacy protect different boundaries.](assets/federated-learning-trust-boundaries.svg){fig-alt="Client updates flow through secure aggregation to a global model and an optional differential privacy layer, with a note that each mechanism addresses a different threat."}

</div>

**Secure aggregation** cryptographically allows a server to learn an aggregate of client updates without seeing each update in the clear. It protects updates from the server under protocol assumptions but does not stop the aggregate or final model from leaking. Client-level DP can clip each client's update and add calibrated noise so the presence of one client has bounded influence. Record-level DP within clients protects a different unit.

<details>
<summary><strong>Python: verify weighted federated averaging and zero-sum masking</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(127)
client_sizes = [120, 240, 80]
clients = []
for client_id, size in enumerate(client_sizes):
    x = rng.normal(loc=0.3 * client_id, scale=1.0, size=size)
    y = 1.8 * x - 0.4 + rng.normal(scale=0.35, size=size)
    clients.append((x, y))

initial = np.array([0.0, 0.0])  # slope and intercept
learning_rate = 0.08

def gradient(parameters, x, y):
    prediction = parameters[0] * x + parameters[1]
    residual = prediction - y
    return np.array([2 * np.mean(residual * x), 2 * np.mean(residual)])

# Every client starts from the same parameters and takes one local step.
local_updates = []
for x, y in clients:
    local_updates.append(-learning_rate * gradient(initial, x, y))

weights = np.array(client_sizes) / sum(client_sizes)
federated_update = np.sum(
    np.array(local_updates) * weights[:, None], axis=0
)

all_x = np.concatenate([client[0] for client in clients])
all_y = np.concatenate([client[1] for client in clients])
central_update = -learning_rate * gradient(initial, all_x, all_y)

# Toy zero-sum masks illustrate the algebra behind secure aggregation.
# Real protocols must handle cryptography, dropouts, and malicious behavior.
mask_0 = rng.normal(size=2)
mask_1 = rng.normal(size=2)
masks = [mask_0, mask_1, -(mask_0 + mask_1)]
masked_updates = [
    client_sizes[k] * local_updates[k] + masks[k]
    for k in range(len(clients))
]
recovered_sum = np.sum(masked_updates, axis=0)
true_sum = np.sum(
    [client_sizes[k] * local_updates[k] for k in range(len(clients))],
    axis=0,
)

print("federated update:", np.round(federated_update, 6))
print("central one-step update:", np.round(central_update, 6))
print("updates match:", np.allclose(federated_update, central_update))
print("masked sum equals true sum:", np.allclose(recovered_sum, true_sum))
print("server-visible masked update 0:", np.round(masked_updates[0], 3))
```

</details>

The equality holds because every client takes one full-batch gradient step from the same parameters and the server weights by client sample count. Multiple local steps on heterogeneous non-IID data no longer equal centralized SGD and can create client drift.

Federated evaluation should report:

- performance and calibration by client and client type;
- convergence under non-IID data and variable participation;
- communication, energy, latency, and dropout robustness;
- poisoning and backdoor resistance;
- server, client, and collusion threat models;
- secure-aggregation assumptions and key management;
- record- or client-level privacy accounting;
- deletion, consent, and model-update policy.

**Summary.** Privacy is an end-to-end property of data collection, learning, interfaces, and release history. Membership tests diagnose particular attacks; differential privacy supplies a formal distributional bound; secure aggregation protects individual updates in transit; and federated learning changes where computation occurs. These mechanisms solve different problems and should be composed under one explicit threat model.


### **Documentation and Governance**

Governance turns technical evidence into accountable decisions. Documentation is useful when it records assumptions, ownership, thresholds, unresolved risks, and changes; it is weak when it becomes a static template completed after deployment.

#### **Datasheets, Model Cards, and Decision Records**

Different artifacts document different objects:

| Artifact | Object documented | Questions it should answer |
|---|---|---|
| Datasheet or data statement | Dataset | Why was it collected? Who is represented? How were consent, labels, missingness, retention, and access handled? |
| Model card | Fitted model and evaluation | What are intended and excluded uses? Which populations and environments were tested? What are the limitations and operating thresholds? |
| System card | End-to-end system | How do model, tools, retrieval, human review, interface, and safeguards interact? |
| Experiment record | Training run | Which code, data snapshot, features, seed, metrics, and artifacts produced this result? |
| Decision record | Consequential choice | Who approved the objective, metric, threshold, mitigation, exception, and residual risk, and why? |
| Monitoring plan | Deployed service | Which signals trigger alert, investigation, rollback, retraining, or retirement? |
| Incident report | Failure event | What happened, who was affected, how was it contained, and what prevents recurrence? |

A model card should not say merely “AUC = 0.91.” It should identify the evaluation population and period, uncertainty, subgroup metrics, shift tests, calibration, decision threshold, abstention policy, known blind spots, privacy guarantees, and prohibited uses. Links to immutable data and code versions make the claim reproducible.

<div class="diagram-scroll">

![Governance is a feedback loop connecting scope, evidence, approval, deployment, and monitoring.](assets/governance-evidence-loop.svg){fig-alt="A lifecycle moves from scope through evidence, decision, deployment, and monitoring, then feeds operational evidence back into the next review."}

</div>

The governance process should include **stage gates**:

1. **Problem gate:** Is machine learning appropriate, and is the decision objective legitimate?
2. **Data gate:** Are provenance, consent, coverage, label validity, and protected-unit definitions adequate?
3. **Evaluation gate:** Does the test design represent deployment, including slices and stress conditions?
4. **Risk gate:** Are interpretation, robustness, fairness, privacy, security, and misuse findings within declared tolerances?
5. **Deployment gate:** Are threshold, fallback, access control, logging, appeal, and rollback implemented?
6. **Monitoring gate:** Are owners, delayed labels, drift tests, incident response, and retirement criteria active?

Approval should not be a single undifferentiated “responsible AI” checkbox. A privacy reviewer, domain owner, security team, affected-community representative, and model developer contribute different knowledge and authority. A decision record should preserve disagreement and residual risk rather than erase it.

Monitoring must follow the causal path from input to impact:

- input schema and support;
- prediction, confidence, and abstention;
- decision threshold and human override;
- latency, failures, and access patterns;
- delayed labels and outcome quality;
- errors, benefits, and burdens by relevant slices;
- appeals, complaints, incidents, and downstream feedback;
- privacy-budget consumption and unusual query behavior.

Drift alerts without an action policy create noise. Each alert needs an owner, window, severity, investigation procedure, and allowed response. Retraining is not always the right response: a concept shift may require a new target or process, a fairness issue may require policy change, and a privacy incident may require containment rather than another model version.

**Summary.** Documentation should make a claim traceable from purpose to evidence to accountable decision. Governance is the operating system that keeps those claims current as data, models, users, and institutions change.


### **Trade-Offs and Responsible Model Choice**

There is no scalar “trustworthiness score” that safely collapses accuracy, calibration, robustness, fairness, privacy, latency, cost, and human impact. These objectives use different units and embody different values. Model selection should therefore use **constraints and a Pareto frontier**, not an arbitrary weighted average hidden inside a leaderboard.

Begin with non-negotiable requirements:

- minimum overall and worst-group task performance;
- maximum calibration error in consequential score regions;
- maximum degradation under named stresses;
- an abstention or fallback capacity the operation can actually handle;
- fairness tolerances matched to a documented harm;
- privacy and security requirements matched to the threat model;
- latency, resource, accessibility, and maintainability constraints.

Among models that pass the gates, prefer the one with the clearest evidence, lower complexity, smaller data footprint, more stable behavior, and easier recovery. A small predictive gain should not automatically outweigh a large loss in auditability or privacy.

| Choice | Possible benefit | Possible cost | Evidence needed |
|---|---|---|---|
| More complex model | Better average fit | Harder audit, calibration, and debugging | Repeated out-of-sample gain and reliable explanation |
| Aggressive augmentation | Corruption robustness | Distorted semantics or subgroup effects | Realistic severity curves and slice checks |
| Fairness constraint | Reduced chosen disparity | Utility shift or another disparity | Frontier, uncertainty, and harm-based justification |
| Stronger DP | Smaller participation leakage | More noise and subgroup utility loss | Privacy accounting and per-group utility |
| More abstention | Lower accepted-case risk | Delayed or denied service, reviewer overload | Risk-coverage and capacity simulation |
| Group-specific threshold | Better error parity | Different treatment and operational complexity | Legal, ethical, and outcome justification |
| Federated architecture | Less raw-data movement | More attack surfaces and client drift | System threat model and client-level evaluation |

A practical reliability audit can follow this sequence:

1. **Define the decision.** Name the prediction, affected people, owner, time horizon, and consequences of each error.
2. **Specify distributions and threats.** State deployment environments, critical subgroups, plausible corruptions, attackers, protected units, and unsupported uses.
3. **Create independent evidence.** Use untouched test data, temporal or external validation, repeated runs, confidence intervals, and delayed outcomes.
4. **Audit learned behavior.** Combine model-specific structure with global and local post-hoc methods; test explanation fidelity and stability.
5. **Stress the system.** Sweep corruption severity, evaluate worst slices, conduct adaptive attacks, and measure risk-coverage under abstention.
6. **Evaluate harms.** Report base rates, confusion components, ranking exposure, calibration, intersections, uncertainty, and the complete decision process.
7. **Evaluate privacy and security.** Test concrete attacks, verify access boundaries, account formal privacy loss, and include every release.
8. **Compare feasible models.** Apply hard gates first, then inspect the Pareto frontier and operational burden.
9. **Document the decision.** Record selected thresholds, rejected alternatives, owners, residual risks, exceptions, and review dates.
10. **Operate and learn.** Monitor outcomes, incidents, appeals, drift, privacy budget, and rollback readiness; retire the system when assumptions fail.

The final question is not “Can this model be explained?” or “Is this model fair?” in the abstract. It is:

> Is this particular model-centered decision process supported by evidence strong enough for this population, environment, threat model, and consequence, and is there an accountable way to detect and repair failure?

Sometimes the responsible choice is an interpretable constrained model, a human-only process, a randomized trial, a rules system, a data-collection change, or no automation at all. Machine learning earns deployment through a bounded, testable claim, not through benchmark performance alone.
